# 02. Optical-Imaging Datasets, Splits, Normalization, and Augmentation

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook explains how optical-imaging data must be organized before deep learning can be scientifically meaningful.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Define the independent biological unit
- Choose 2-D/2.5-D/3-D appropriately
- Select normalization without destroying information
- Avoid patch-level leakage
- Design plausible augmentation
- Understand paired versus unpaired training


## Mind map

```mermaid
mindmap
  root((Optical dataset))
    Biological unit
      Patient
      Animal
      Tissue
    Representation
      2D
      2.5D
      3D
    Preprocessing
      Normalization
      Patches
      Augmentation
    Pairing
      Paired
      Registered
      Unpaired
    Splits
      Train
      Validation
      Test
      External

```


## 1. Why data design comes before model design

In biomedical imaging, a sophisticated architecture cannot rescue a scientifically invalid dataset split.

First define the **independent unit**. Examples:

| Modality / experiment | Likely independent unit |
|---|---|
| OCT retina | patient/eye |
| mouse embryo imaging | embryo/dam depending design |
| LSFM zebrafish | fish |
| MUSE tissue | patient/tissue block |
| pathology patches | patient/slide/block depending question |

Patches, tiles, z-slices, repeated frames, or augmentations from one unit are correlated observations.


## 2. 2-D, 2.5-D, or 3-D?

Choose based on the scientific information and resources.

- **2-D:** easiest baseline; each slice independent.
- **2.5-D:** neighboring slices/channels are stacked as input channels.
- **3-D:** uses volumetric context but needs much more memory/data.

Do not use 3-D only because the original data are 3-D. If z-resolution is highly anisotropic or training data are small, a 2-D/2.5-D baseline may be more stable.


## 3. Normalization choices

Common options:

1. fixed physical/intensity scaling;
2. global dataset mean/std;
3. per-image z-score;
4. percentile scaling;
5. log transform + scaling for high-dynamic-range signals.

### Important
Per-image normalization can remove absolute-intensity information. That may be good for morphology-only segmentation but harmful if absolute signal strength is biologically meaningful.


In [ ]:
import torch

image = torch.tensor([[0., 10., 20.],
                      [30., 40., 1000.]])

# Percentile-like robust scaling would normally use quantiles.
lo = torch.quantile(image, 0.01)
hi = torch.quantile(image, 0.99)
scaled = (image - lo) / (hi - lo + 1e-8)
scaled = torch.clamp(scaled, 0, 1)

print("lo, hi:", lo.item(), hi.item())
print(scaled)


## 4. Patch extraction: useful but dangerous

Patches help with memory and increase the number of training samples. But they do **not create new independent subjects**.

Correct order:

```text
specimens
  ↓
split specimens into train/val/test
  ↓
extract patches independently within each split
```

Dangerous order:

```text
extract thousands of patches from all specimens
  ↓
randomly split patches
```

The second workflow can put nearly identical neighboring patches from one specimen into both train and test.


## 5. Augmentation must preserve the label

An augmentation is valid only if the transformed input still corresponds to the transformed target and remains physically/biologically plausible.

Often reasonable:
- flips/rotations when orientation is irrelevant;
- mild intensity scaling if acquisition variation is realistic;
- crop/translation;
- mild noise consistent with the modality.

Potentially harmful:
- vertical flip when anatomy has a meaningful superior/inferior direction;
- arbitrary hue changes in histology/virtual staining;
- elastic deformation that changes pathology;
- noise model unrelated to OCT/fluorescence physics.


## 6. Paired versus unpaired data

**Paired:** each input has a corresponding target of the same scene/specimen, ideally registered.  
Examples: low-SNR → high-SNR fluorescence; MUSE → registered H&E.

**Unpaired:** source and target distributions exist but individual images do not correspond.

If a paper claims pixel-level fidelity but training pairs are poorly registered, a pixel loss can punish correct structures simply because they are shifted.


## 7. Dataset class mental model

PyTorch's `Dataset` answers two questions:

- `__len__`: how many examples?
- `__getitem__(i)`: what are input and target for example `i`?

The dataset should make preprocessing explicit and deterministic for validation/test.


In [ ]:
from torch.utils.data import Dataset
import torch

class ToyOpticalDataset(Dataset):
    def __init__(self, n=10):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, index):
        # Synthetic example: 1x64x64 image and binary mask
        image = torch.rand(1, 64, 64)
        mask = (image > 0.6).float()
        return image, mask

dataset = ToyOpticalDataset()
image, mask = dataset[0]
print(image.shape, mask.shape)


## 8. Data audit before training

Make a table for every dataset:

| Question | Answer |
|---|---|
| number of independent specimens | ? |
| images/volumes per specimen | ? |
| channels | ? |
| pixel size / z spacing | ? |
| intensity dtype/range | ? |
| missing/corrupt files | ? |
| label source | ? |
| class balance | ? |
| split unit | ? |
| train/val/test counts by specimen | ? |
| acquisition sites/instruments | ? |

If these answers are unknown, model development is premature.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
